Metrics (8)

- pos_adv_noun — adv / noun count
- pos_adv_pron — adv / pronoun count
- pos_adj_verb — adj / verb count
- pos_noun_verb — noun / verb count
- pos_verb_pron — verb / pronoun count
- pos_adv_adj — adv / adj count
- pos_adj_pron — adj / pronoun count
- pos_noun_pron — noun / pronoun count

In [ ]:
import sys
import importlib
import spacy

importlib.invalidate_caches() # see packages installed after the kernel started
# download only if missing (uses the kernel's own python)
if not spacy.util.is_package("en_core_web_trf"):
    !{sys.executable} -m spacy download en_core_web_trf
print("en_core_web_trf installed:", spacy.util.is_package("en_core_web_trf"))

In [ ]:
import spacy
from spacy.tokens import Doc

# en_core_web_trf - same parser as mean-mdl.ipynb and kazmi-syntactic.ipynb, so sentences and POS tags come from the same parse
# only pos_ (tagger + attribute_ruler) and sents (parser) are needed, so ner and lemmatizer are disabled
spacy.prefer_gpu() # uses the Mac GPU (MPS) if available
nlp = spacy.load("en_core_web_trf", disable=["ner", "lemmatizer"])
print(f"Language model: {nlp.meta['lang']}_{nlp.meta['name']}")
print(f"Version: {nlp.meta['version']}")
print(f"Pipe names: {nlp.pipe_names}")

feature_names = [
    "pos_adv_noun",
    "pos_adv_pron",
    "pos_adj_verb",
    "pos_noun_verb",
    "pos_verb_pron",
    "pos_adv_adj",
    "pos_adj_pron",
    "pos_noun_pron",
]

def calculate_kazmi_pos(text):
    text = text or ""
    result = {}
    for name in feature_names:
        result[name] = f"{0:.5f}"

    #text can be a raw string or an already parsed doc (from nlp.pipe), so the paragraph is never parsed twice
    doc = text if isinstance(text, Doc) else nlp(text)
    n_adv = 0
    n_noun = 0
    n_pron = 0
    n_adj = 0
    n_verb = 0
    for token in doc:
        if token.pos_ == "ADV":
            n_adv += 1
        elif token.pos_ == "NOUN" or token.pos_ == "PROPN":
            n_noun += 1
        elif token.pos_ == "PRON":
            n_pron += 1
        elif token.pos_ == "ADJ":
            n_adj += 1
        elif token.pos_ == "VERB":
            n_verb += 1

    if n_noun != 0:
        result["pos_adv_noun"] = f"{(n_adv / n_noun):.5f}"
    if n_pron != 0:
        result["pos_adv_pron"] = f"{(n_adv / n_pron):.5f}"
        result["pos_verb_pron"] = f"{(n_verb / n_pron):.5f}"
        result["pos_adj_pron"] = f"{(n_adj / n_pron):.5f}"
        result["pos_noun_pron"] = f"{(n_noun / n_pron):.5f}"
    if n_verb != 0:
        result["pos_adj_verb"] = f"{(n_adj / n_verb):.5f}"
        result["pos_noun_verb"] = f"{(n_noun / n_verb):.5f}"
    if n_adj != 0:
        result["pos_adv_adj"] = f"{(n_adv / n_adj):.5f}"

    return result

In [ ]:
from pathlib import Path
import csv

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/raw-csv to read and data/<LEVEL>/features/ to write
raw_csv_path = Path(f"../../data/{LEVEL}/raw-csv")
# old values (en_core_web_sm) are in archive-pooled-sm-2026-11-11/kazmi-pos-csv
out_csv_path = Path(f"../../data/{LEVEL}/features/kazmi-pos-csv")
out_csv_path.mkdir(parents=True, exist_ok=True)

for csv_file in sorted(raw_csv_path.glob("*.csv")):
    out_csv_file = out_csv_path / csv_file.name
    if out_csv_file.exists() and out_csv_file.stat().st_size > 0: # Skip if already exist
       continue
    print(f"Working on file: {csv_file.name}")
    with csv_file.open("r",encoding="utf-8") as cf:
       csv_reader = csv.DictReader(cf)
       column_names = list(csv_reader.fieldnames)
       column_names = column_names[:-1] + feature_names + [column_names[-1]]
       rows = list(csv_reader)
    # write to a temp file and rename at the end, so a half written file is never skipped on restart
    tmp_csv_file = out_csv_file.with_suffix(".tmp")
    with tmp_csv_file.open("w",encoding="utf-8") as of:
       csv_writer = csv.DictWriter(of,fieldnames=column_names)
       csv_writer.writeheader()
       # nlp.pipe parses texts in batches - much faster than calling nlp(text) for every row
       for row, doc in zip(rows, nlp.pipe([row["text"] or "" for row in rows], batch_size=64)):
          feats = calculate_kazmi_pos(doc)
          for name in feature_names:
             row[name] = feats[name]
          csv_writer.writerow(row)
    tmp_csv_file.replace(out_csv_file)
    #break - During demo we can uncomment and providea demo path

In [ ]:
# TEST KAZMI POS CODE

text = "Use a dehydrator to dry them if you have one. Set your dehydrator to 150 F degrees. Lay the almonds in a single layer on the dehydrator trays. Place the almonds in the dehydrator for 12 to 24 hours, or until crispy."
print(calculate_kazmi_pos(text))
text = "Beyoncé and Rihanna are tied for the second - most #1 entries on the Billboard Hot 100 chart, with 5 each."
print(calculate_kazmi_pos(text))